# NESCAC Ice Hockey (M) Principal Component Analysis
## 2022-23 through 2025-26 standings and team stats

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import matplotlib.pyplot as plt
import pandas as pd
import glob


In [ ]:
# Find all CSV files in the current folder
csv_files = glob.glob('*.csv')

# Load the first four CSV files
drop_cols = ['Team','Neutral','Neutral Att','Neutral Att Avg','All Att','All Att Avg','All']

df1 = pd.read_csv(csv_files[0]).drop(columns=drop_cols)
df2 = pd.read_csv(csv_files[1]).drop(columns=drop_cols)
df3 = pd.read_csv(csv_files[2]).drop(columns=drop_cols)
df4 = pd.read_csv(csv_files[3]).drop(columns=drop_cols)


## Clean the data, clean clean the data!

In [ ]:
# Split combined columns into separate numeric columns
for df in [df1, df2, df3, df4]:
    df[['GF', 'GA']] = df['GF-GA'].str.split(' - ', expand=True).astype(int)
    df.drop(columns=['GF-GA'], inplace=True)

for df in [df1, df2, df3, df4]:
    df[['AVGH', 'AVGV']] = df['AVG Score'].str.split(' - ', expand=True).astype(float)
    df.drop(columns=['AVG Score'], inplace=True)

for df in [df1, df2, df3, df4]:
    df[['Pen Num', 'Pen Min']] = df['Pen'].str.split(' - ', expand=True).astype(str)
    df.drop(columns=['Pen'], inplace=True)

for df in [df1, df2, df3, df4]:
    df['Pen Num'] = df['Pen Num'].str.replace('None', '0').astype(int)
    df['Pen Min'] = df['Pen Min'].str.replace(':00', '').astype(int)


In [ ]:
df1.head()

## Standardize the data before dimension reduction

PCA and t-SNE both depend on distances or variation. Since `Away Att`, `Shots`, `Sav%`, and `Shot%` are on very different scales, we first convert each feature to a z-score.

This means each column is measured in standard deviations from its mean.

In [ ]:
df1['champion'] = (df1['Standing'] == 1)

# Use numeric hockey/statistical features, but do not use Standing or champion as inputs
features = [
    col for col in df1.select_dtypes(include='number').columns
    if col not in ['Standing', 'champion']
]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df1[features])

features

## Sample means and variances

These summaries help us notice why scaling matters before PCA.

In [ ]:
means = df1.mean(numeric_only=True)
print('Sample Means:')
print(means)


In [ ]:
variances = df1.var(numeric_only=True)
print('\nSample Variances:')
print(variances)

## PCA computation

Now we run PCA on the standardized feature matrix.

In [ ]:
pca = PCA(n_components=3)
scores = pca.fit_transform(X_scaled)

pca.explained_variance_ratio_

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=features,
    columns=['PC1', 'PC2', 'PC3']
)

loadings

In [ ]:
plt.scatter(scores[:,0], scores[:,1], c=df1['Standing'], cmap='viridis')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.colorbar(label='Standing')
plt.title('PCA projection of NESCAC hockey teams')
plt.show()

# Side Quest: t-SNE Projection

PCA asks:

> Which directions explain the most variation in the data?

t-SNE asks a different question:

> Which teams are near each other, and can we draw a 2D picture that preserves those local neighborhoods?

So PCA is about **variance**. t-SNE is about **neighbors**.

## Distances become neighborhoods

Imagine three teams: A, B, and C.

If B is very close to A in the full statistical data, while C is far away, then t-SNE tries to keep A and B close together in the 2D picture.

But t-SNE does **not** try to preserve every exact distance. It mainly tries to preserve which points are nearby.

## Important warning

In a t-SNE plot:

- nearby points are meaningful,
- clusters can be meaningful,
- the axes do not have natural interpretations,
- large distances between clusters should be interpreted cautiously,
- the shape and size of clusters can be misleading.

This is very different from PCA, where PC1 and PC2 are specific linear combinations of the original variables.

In [ ]:
tsne = TSNE(
    n_components=2,
    perplexity=3,
    learning_rate='auto',
    init='pca',
    random_state=42
)

tsne_scores = tsne.fit_transform(X_scaled)

In [ ]:
plt.scatter(tsne_scores[:,0], tsne_scores[:,1], c=df1['Standing'], cmap='viridis')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.colorbar(label='Standing')
plt.title('t-SNE projection of NESCAC hockey teams')
plt.show()

## Reflection questions

1. Which teams appear close together in the PCA plot?
2. Which teams appear close together in the t-SNE plot?
3. Are the same teams close together in both plots?
4. Which plot seems better for finding clusters of similar teams?
5. Which plot seems better for explaining which variables matter most?
6. Why should we be careful about interpreting the axes in the t-SNE plot?